# 00 — 6D merged output audit

Check the inventory, metadata, axes, and stored values of the new 6D campaign against the frozen schema in the [expanded reference](Past_reference/ABCD_ExpandedPlane_Scan.ipynb). This notebook does not define physics cuts, ABCD regions, working points (WPs), or closure tests.

The default payload audit covers **18 background samples and 4 reference signal samples**. For data, check filenames and file sizes only; do not open the payloads. Inspect the stored sumw/sumw2 without applying luminosity or cross-section factors again. Passing the structural checks does not constitute an independent validation of the production normalization.
The current metadata YAML records the selection definition as `<not found>` and the histogram collection as `hists: []`. Therefore, schema validation here is separate from verifying the provenance of the frozen baseline cuts. The baseline selection definition must be verified separately before subsequent plane analyses.


In [1]:
from pathlib import Path
import gc
import sys

SIDM_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "sidm"), None)
if SIDM_DIR is None:
    raise RuntimeError("Run this notebook from within the sidm repository.")
if str(SIDM_DIR.parent) not in sys.path:
    sys.path.insert(0, str(SIDM_DIR.parent))

# Set the LPC NumPy 2 pickle compatibility aliases after importing pyarrow.
import coffea.util
import numpy as np
import pandas as pd
from IPython.display import display

if int(np.__version__.split(".")[0]) < 2:
    import numpy.core as _numpy_core
    import numpy.core.numeric as _numpy_numeric
    sys.modules.setdefault("numpy._core", _numpy_core)
    sys.modules.setdefault("numpy._core.numeric", _numpy_numeric)

## Inputs and scope

Read the `ABCD_6D_inclusive_eHEM_skimv1_v1` merged samples copied into this directory. Change `INPUT_BASE` to use another location. Do not substitute the previous 2D campaign.

The expected counts are 18 background files, 68 data files, and 180 signal files. Checking file counts does not validate the complete list of names against an independent production manifest. Set `CHECK_ALL_SIGNALS=True` to inspect all signal samples sequentially. CSV export is disabled by default.

In [2]:
CAMPAIGN = "ABCD_6D_inclusive_eHEM_skimv1_v1"
INPUT_BASE = SIDM_DIR / "studies/ABCD_plane_summary"
EXPECTED_FILES = {"bkg": 18, "data": 68, "signal": 180}
CHECK_ALL_SIGNALS = False
EXPORT_CSV = False
EXPORT_DIR = SIDM_DIR / "studies/ABCD_plane_summary/6D_output_audit"
TOPOLOGIES = ("2mu2e", "4mu")
VARS = ("iso0", "iso1", "disp0", "disp1", "abs_dphi", "ljlj_mass")
SIGNAL_REFERENCES = (
    "2Mu2E_1000GeV_0p25GeV_0p002mm", "2Mu2E_1000GeV_0p25GeV_0p2mm",
    "4Mu_1000GeV_0p25GeV_0p002mm", "4Mu_1000GeV_0p25GeV_0p2mm",
)
REFERENCE_PAIRS = ("iso0_vs_iso1", "iso0_vs_abs_dphi", "disp0_vs_disp1")
EDGES = {
    "iso": np.array([0, .05, .1, .15, .2, .25, .3, .4, .5, .75, 1, 2, 5]),
    "pixel": np.arange(-.5, 12, 1, dtype=float),
    "lost": np.arange(-.5, 10, 1, dtype=float),
    "abs_dphi": np.array([0, .5, 1, 1.5, 2, 2.2, 2.4, 2.6, 2.7, 2.8, 2.9, 3, 3.05, 3.1, np.pi]),
    "ljlj_mass": np.array([0, 50, 100, 150, 200, 300, 400, 600, 800, 1000, 1500, 2500], float),
}

## Shared helpers

Define functions before the cells that use them. Check the exact histogram key set (two 6D histograms and six reference histograms), Weight storage, finite sumw/sumw2, and nonnegative sumw2. For the 6D histograms, also check the single categorical channel, axis order, flow bins, and frozen bin edges.

Avoid copying large cubes or retaining multiple samples in memory. Keep only scalar totals and release each payload. Do not sum different signal hypotheses. The known good_matched_muons entry is not inspected separately.

Also compare sumw and sumw2 in the six reference histograms with direct projections of the 6D histograms (including flow bins, rtol=1e-10, atol=1e-9). This comparison checks internal consistency within the same output.


In [3]:
def axis_edges(topology, variable):
    if variable in ("iso0", "iso1"):
        kind = "iso"
    elif variable in ("disp0", "disp1"):
        kind = "lost" if topology == "2mu2e" and variable == "disp1" else "pixel"
    else:
        kind = variable
    return EDGES[kind]


def collect_inventory(source):
    directory = INPUT_BASE / f"{CAMPAIGN}_{source}_merged_samples_v1"
    if not directory.is_dir():
        raise FileNotFoundError(f"Input directory not found: {directory}")
    return [dict(source=source, sample=path.stem, filename=path.name,
                 bytes=path.stat().st_size, location=str(path),
                 metadata_file=(path.with_suffix(".meta.yaml")).is_file())
            for path in sorted(directory.glob("*.coffea"))]


def load_mc_payload(record):
    if record["source"] not in {"bkg", "signal"}:
        raise ValueError("Loading data payloads is not permitted.")
    return coffea.util.load(record["location"])


def inspect_mc_payload(payload, sample, source):
    if source not in {"bkg", "signal"}:
        raise ValueError("Only MC payloads are inspected.")
    if not isinstance(payload, dict) or set(payload.get("out", {})) != {sample}:
        raise ValueError(f"Sample identity mismatch: {sample}")
    out = payload["out"][sample]
    core_names = {f"abcd6d_{t}" for t in TOPOLOGIES}
    expected = core_names | {
        f"abcd6d_ref_{t}_{pair}" for t in TOPOLOGIES for pair in REFERENCE_PAIRS
    }
    if set(out.get("hists", {})) != expected:
        raise ValueError("6D schema mismatch: two abcd6d and six reference histograms are required. "
                         "The previous 2D campaign cannot be used.")
    metadata = out["metadata"]
    if set(metadata["is_data"]) != {False}:
        raise ValueError("is_data metadata mismatch")
    if set(metadata["unweighted_hist"]) != {False}:
        raise ValueError("Weighted histogram metadata is required.")
    if {str(value) for value in metadata["year"]} != {"2018"}:
        raise ValueError("Year metadata mismatch")
    histogram_rows, axis_rows = [], []
    for name, histogram in out["hists"].items():
        if histogram.storage_type.__name__ != "Weight":
            raise ValueError(f"Weight storage mismatch: {name}")
        weights, variances = histogram.values(flow=True), histogram.variances(flow=True)
        if variances is None or weights.shape != variances.shape:
            raise ValueError(f"Missing sumw2 or shape mismatch: {name}")
        if not np.isfinite(weights).all() or not np.isfinite(variances).all():
            raise ValueError(f"Finite storage check failed: {name}")
        if np.any(variances < 0):
            raise ValueError(f"Negative sumw2: {name}")
        topology = name.removeprefix("abcd6d_") if name in core_names else None
        row = dict(source=source, sample=sample, histogram=name, topology=topology,
                   histogram_kind="6D" if topology else "reference", status="VERIFIED",
                   storage="Weight", shape_flow=str(weights.shape),
                   sumw=np.nan, sumw2=np.nan, neff=np.nan)
        if topology is not None:
            if tuple(axis.name for axis in histogram.axes) != ("channel",) + VARS:
                raise ValueError(f"Axis name/order mismatch: {name}")
            channel_axis = histogram.axes["channel"]
            channel = f"abcd6d_inclusive_{topology}_dxySpread_cosAlpha_mu_veto_eHEM"
            if (not channel_axis.traits.discrete or channel_axis.traits.underflow
                    or channel_axis.traits.overflow or list(channel_axis) != [channel]):
                raise ValueError(f"Single categorical channel mismatch: {name}")
            for variable in VARS:
                axis, expected_edges = histogram.axes[variable], axis_edges(topology, variable)
                if not (axis.traits.underflow and axis.traits.overflow):
                    raise ValueError(f"Missing flow bins: {name}/{variable}")
                if np.shape(axis.edges) != expected_edges.shape or not np.allclose(
                        axis.edges, expected_edges, rtol=0, atol=1e-12):
                    raise ValueError(f"Frozen bin edge mismatch: {name}/{variable}")
                axis_rows.append(dict(source=source, sample=sample, topology=topology,
                                      axis=variable, bins=axis.size, underflow=True, overflow=True,
                                      edges=tuple(float(x) for x in axis.edges)))
            expected_shape = (1,) + tuple(len(axis_edges(topology, v)) + 1 for v in VARS)
            if weights.shape != expected_shape:
                raise ValueError(f"Shape mismatch including flow bins: {name}")
            sumw, sumw2 = float(weights.sum()), float(variances.sum())
            if not np.isfinite(sumw) or not np.isfinite(sumw2):
                raise ValueError(f"Nonfinite totals: {name}")
            row.update(sumw=sumw, sumw2=sumw2, neff=sumw**2 / sumw2 if sumw2 > 0 else np.nan)
        histogram_rows.append(row)
    # Compare 2D projections of the 6D histograms, including flow bins, with the references stored during production.
    for topology in TOPOLOGIES:
        for pair in REFERENCE_PAIRS:
            name = f"abcd6d_ref_{topology}_{pair}"
            reference = out["hists"][name]
            projected = out["hists"][f"abcd6d_{topology}"].project("channel", *pair.split("_vs_"))
            if tuple(axis.name for axis in projected.axes) != tuple(axis.name for axis in reference.axes):
                raise ValueError(f"Reference projection axis mismatch: {name}")
            for observed, expected_axis in zip(reference.axes, projected.axes):
                if observed != expected_axis:
                    raise ValueError(f"Reference projection binning mismatch: {name}")
            for method in ("values", "variances"):
                observed = getattr(reference, method)(flow=True)
                expected_values = getattr(projected, method)(flow=True)
                if observed.shape != expected_values.shape or not np.allclose(
                        observed, expected_values, rtol=1e-10, atol=1e-9):
                    raise ValueError(f"reference projection {method} mismatch: {name}")
            for row in histogram_rows:
                if row["histogram"] == name:
                    row["projection_match"] = True
    return histogram_rows, axis_rows


def audit_records(records):
    file_rows, histogram_rows, axis_rows = [], [], []
    for index, record in enumerate(records, 1):
        payload = None
        try:
            payload = load_mc_payload(record)
            histograms, axes = inspect_mc_payload(payload, record["sample"], record["source"])
            histogram_rows.extend(histograms)
            axis_rows.extend(axes)
            file_rows.append(dict(source=record["source"], sample=record["sample"],
                                  status="VERIFIED", message="Schema/storage checks passed"))
        except Exception as error:
            file_rows.append(dict(source=record["source"], sample=record["sample"],
                                  status="UNVERIFIED", message=str(error)))
        finally:
            payload = None
            gc.collect()
        print(f"MC audit {index}/{len(records)}: {record['sample']} — {file_rows[-1]['status']}")
    return (pd.DataFrame(file_rows, columns=["source", "sample", "status", "message"]),
            pd.DataFrame(histogram_rows), pd.DataFrame(axis_rows))

## 1. File inventory

Check the expected counts, file sizes, duplicate names, and presence of metadata YAML files. Proceed to the MC payload audit only for sources that pass the count, size, and duplicate-name checks. For data, check filenames and file status only.

In [4]:
print(f"Input location: {INPUT_BASE}")
inventory_records, inventory_status_rows = [], []
for source, expected_count in EXPECTED_FILES.items():
    try:
        records = collect_inventory(source)
        inventory_records.extend(records)
        passed = (len(records) == expected_count and bool(records)
                  and all(record["bytes"] > 0 for record in records)
                  and len({record["sample"] for record in records}) == len(records))
        inventory_status_rows.append(dict(
            source=source, expected_files=expected_count, found_files=len(records),
            total_bytes=sum(record["bytes"] for record in records),
            metadata_files=sum(record["metadata_file"] for record in records),
            status="VERIFIED" if passed else "UNVERIFIED",
            message="Counts, sizes, and duplicate names checked" if passed else "Count, size, or duplicate-name mismatch"))
    except Exception as error:
        inventory_status_rows.append(dict(
            source=source, expected_files=expected_count, found_files=pd.NA,
            total_bytes=pd.NA, metadata_files=pd.NA, status="UNVERIFIED", message=str(error)))
inventory = pd.DataFrame(inventory_records,
                         columns=["source", "sample", "filename", "bytes", "location", "metadata_file"])
inventory_status = pd.DataFrame(inventory_status_rows)
display(inventory_status)
print("Filenames and sizes are retained in the inventory DataFrame. Data payloads were not opened.")

Input location: /home/cms-jovyan/workspace/CMS-SIDM-PR/sidm/studies/ABCD_plane_summary


,source,expected_files,found_files,total_bytes,metadata_files,status,message
0,bkg,18,18,18070002,18,VERIFIED,"Counts, sizes, and duplicate names checked"
1,data,68,68,67085090,68,VERIFIED,"Counts, sizes, and duplicate names checked"
2,signal,180,180,184717727,180,VERIFIED,"Counts, sizes, and duplicate names checked"


Filenames and sizes are retained in the inventory DataFrame. Data payloads were not opened.


## 2. Background and signal payload audit

By default, inspect all background samples and the four reference signal samples. If a reference sample is missing, leave the corresponding signal audit unverified. The inclusive totals below audit stored values including flow bins; they are not yields or efficiencies for a physics selection.


In [5]:
verified_sources = set(inventory_status.loc[inventory_status.status.eq("VERIFIED"), "source"])
requested_counts = {"bkg": 18, "signal": 180 if CHECK_ALL_SIGNALS else len(SIGNAL_REFERENCES)}
selected_records, selection_messages = [], {}
for source in ("bkg", "signal"):
    if source not in verified_sources:
        selection_messages[source] = "Input inventory unverified: payload audit skipped"
        continue
    candidates = [record for record in inventory_records if record["source"] == source]
    if source == "signal" and not CHECK_ALL_SIGNALS:
        missing = set(SIGNAL_REFERENCES) - {record["sample"] for record in candidates}
        if missing:
            selection_messages[source] = "Missing reference samples: " + ", ".join(sorted(missing))
            continue
        candidates = [record for record in candidates if record["sample"] in SIGNAL_REFERENCES]
    selected_records.extend(candidates)

file_audit, histogram_audit, axis_audit = audit_records(selected_records)
display(file_audit)
if not axis_audit.empty:
    print("Verified 6D axis schema (deduplicated across samples):")
    display(axis_audit.drop(columns=["source", "sample"]).drop_duplicates())
if not histogram_audit.empty:
    print("MC inclusive stored-value audit (including flow bins; each signal hypothesis treated independently):")
    display(histogram_audit.loc[histogram_audit.histogram_kind.eq("6D"),
                               ["source", "sample", "topology", "sumw", "sumw2", "neff"]])


MC audit 1/22: DYJetsToLL_M10to50 — VERIFIED


MC audit 2/22: DYJetsToLL_M50 — VERIFIED


MC audit 3/22: QCD_Pt1000 — VERIFIED


MC audit 4/22: QCD_Pt120To170 — VERIFIED


MC audit 5/22: QCD_Pt15To20 — VERIFIED


...

,source,sample,status,message
0,bkg,DYJetsToLL_M10to50,VERIFIED,Schema/storage checks passed
1,bkg,DYJetsToLL_M50,VERIFIED,Schema/storage checks passed
2,bkg,QCD_Pt1000,VERIFIED,Schema/storage checks passed
3,bkg,QCD_Pt120To170,VERIFIED,Schema/storage checks passed
4,bkg,QCD_Pt15To20,VERIFIED,Schema/storage checks passed
5,bkg,QCD_Pt170To300,VERIFIED,Schema/storage checks passed
6,bkg,QCD_Pt20To30,VERIFIED,Schema/storage checks passed
7,bkg,QCD_Pt300To470,VERIFIED,Schema/storage checks passed
8,bkg,QCD_Pt30To50,VERIFIED,Schema/storage checks passed
9,bkg,QCD_Pt470To600,VERIFIED,Schema/storage checks passed


Verified 6D axis schema (deduplicated across samples):


,topology,axis,bins,underflow,overflow,edges
0,2mu2e,iso0,12,True,True,"(0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0...."
1,2mu2e,iso1,12,True,True,"(0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0...."
2,2mu2e,disp0,12,True,True,"(-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5,..."
3,2mu2e,disp1,10,True,True,"(-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5,..."
4,2mu2e,abs_dphi,14,True,True,"(0.0, 0.5, 1.0, 1.5, 2.0, 2.2, 2.4, 2.6, 2.7, ..."
5,2mu2e,ljlj_mass,11,True,True,"(0.0, 50.0, 100.0, 150.0, 200.0, 300.0, 400.0,..."
6,4mu,iso0,12,True,True,"(0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0...."
7,4mu,iso1,12,True,True,"(0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0...."
8,4mu,disp0,12,True,True,"(-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5,..."
9,4mu,disp1,12,True,True,"(-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5,..."


MC inclusive stored-value audit (including flow bins; each signal hypothesis treated independently):


,source,sample,topology,sumw,sumw2,neff
0,bkg,DYJetsToLL_M10to50,2mu2e,513.987371,4.892278e+03,54.000000
1,bkg,DYJetsToLL_M10to50,4mu,38.073139,3.623910e+02,4.000000
8,bkg,DYJetsToLL_M50,2mu2e,451.477495,1.466417e+03,139.000000
9,bkg,DYJetsToLL_M50,4mu,344.292190,1.118275e+03,106.000000
16,bkg,QCD_Pt1000,2mu2e,10.858580,3.670883e-02,3212.000000
17,bkg,QCD_Pt1000,4mu,19.938949,6.740619e-02,5898.000000
24,bkg,QCD_Pt120To170,2mu2e,6485.886576,4.572470e+05,92.000000
25,bkg,QCD_Pt120To170,4mu,3806.933425,2.683841e+05,54.000000
32,bkg,QCD_Pt15To20,2mu2e,0.000000,0.000000e+00,NaN
33,bkg,QCD_Pt15To20,4mu,0.000000,0.000000e+00,NaN


## 3. Validation scope

VERIFIED refers only to the schema/storage checks for the requested files. Provide background totals only if all 18 samples pass. neff = sumw²/sumw2 is a weight diagnostic. This notebook does not assess signal normalization or the validity of the physics analysis. Data schemas and numerical values remain unverified.


In [6]:
scope_rows = []
for source in ("bkg", "signal"):
    checked = file_audit.loc[file_audit.source.eq(source)]
    passed = int(checked.status.eq("VERIFIED").sum())
    complete = passed == requested_counts[source] and len(checked) == requested_counts[source]
    scope_rows.append(dict(
        source=source, requested=requested_counts[source], verified=passed,
        status="VERIFIED" if complete else "UNVERIFIED",
        message=selection_messages.get(source, "Requested MC schema/storage checks completed"
                                       if complete else "Some or all payloads remain unverified")))
scope_rows.append(dict(source="data", requested=0, verified=0, status="NOT_CHECKED",
                       message="File inventory checked only; payloads, schemas, and numerical values were not accessed"))
audit_scope = pd.DataFrame(scope_rows)
display(audit_scope)
background_totals = pd.DataFrame(columns=["topology", "sumw", "sumw2", "neff"])
if audit_scope.loc[audit_scope.source.eq("bkg"), "status"].iloc[0] == "VERIFIED":
    background_totals = (histogram_audit.loc[
        histogram_audit.source.eq("bkg") & histogram_audit.histogram_kind.eq("6D")
    ].groupby("topology")[["sumw", "sumw2"]].sum().reset_index())
    background_totals["neff"] = np.where(
        background_totals.sumw2 > 0, background_totals.sumw**2 / background_totals.sumw2, np.nan)
    display(background_totals)
if audit_scope.loc[audit_scope.source.isin(["bkg", "signal"]), "status"].eq("UNVERIFIED").any():
    print("Unverified items remain. Do not interpret this run as complete validation of the original 6D output.")
print("Independent normalization validation: not performed. Physics cuts/WPs/ABCD closure: not defined.")


,source,requested,verified,status,message
0,bkg,18,18,VERIFIED,Requested MC schema/storage checks completed
1,signal,4,4,VERIFIED,Requested MC schema/storage checks completed
2,data,0,0,NOT_CHECKED,"File inventory checked only; payloads, schemas, and numerical values were not accessed"


,topology,sumw,sumw2,neff
0,2mu2e,25944.804083,2.306726e+06,291.813153
1,4mu,15694.809881,1.917828e+06,128.440614


Independent normalization validation: not performed. Physics cuts/WPs/ABCD closure: not defined.


## Optional table export

Save audit tables only when EXPORT_CSV=True. Do not save payloads or numerical data values. The status of an unverified run is still recorded in inventory_status.csv and audit_scope.csv.


In [7]:
if EXPORT_CSV:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    tables = dict(inventory=inventory, inventory_status=inventory_status,
                  file_audit=file_audit, histogram_audit=histogram_audit,
                  axis_audit=axis_audit, audit_scope=audit_scope, background_totals=background_totals)
    for name, frame in tables.items():
        frame.to_csv(EXPORT_DIR / f"{name}.csv", index=False)
    print(f"Audit tables saved: {EXPORT_DIR}")
else:
    print("CSV export disabled: inspect the DataFrames in memory.")


CSV export disabled: inspect the DataFrames in memory.
